# Causal Analysis Outline
This notebook seeds the Release 1.0 causal analysis workflow. It captures
datasets, modelling ideas, and validation hooks that will be refined with
research stakeholders before code is productionised.

## Working Checklist
- [ ] Assemble joined datasets (entropy signals, returns, controls).- [ ] Explore descriptive statistics and stability diagnostics.- [ ] Prototype causal estimators (difference-in-differences, IV, synthetic control).- [ ] Run sensitivity analyses (placebo windows, robustness to controls).- [ ] Export summary tables and plots for documentation and dashboards.

## Next Steps
1. Mirror methodology updates into `docs/causal_analysis_plan.md` so code,
   notebooks, and documentation stay aligned.
2. Use this notebook as the staging ground for experiments before promoting
   stable utilities into the `entropy_news.research.causal` package.

## Updated workflow

This outline notebook now mirrors the production toolkit shipped in `entropy_news.research.causal`. The steps below assemble a synthetic panel, estimate causal effects, and generate reporting artefacts.

In [ ]:
import pandas as pd
from entropy_news.research.causal import (
    CausalPanelConfig,
    assemble_causal_panel,
    build_propensity_features,
    difference_in_differences,
    two_stage_least_squares,
    synthetic_control,
    build_summary_table,
    PolicyScenario,
    format_policy_narrative,
)

entropy = pd.DataFrame(
    {
        "unit": ["treated", "control_a", "control_b"] * 3,
        "time": [0, 0, 0, 1, 1, 1, 2, 2, 2],
        "treatment": [1, 0, 0, 1, 0, 0, 1, 0, 0],
        "post": [0, 0, 0, 1, 1, 1, 1, 1, 1],
        "instrument": [0.4, 0.2, 0.2, 1.2, 0.3, 0.3, 1.2, 0.4, 0.4],
    }
)
market = pd.DataFrame(
    {
        "unit": ["treated", "control_a", "control_b"] * 3,
        "time": [0, 0, 0, 1, 1, 1, 2, 2, 2],
        "outcome": [10.0, 10.0, 10.2, 14.0, 10.7, 11.05, 15.0, 11.4, 11.9],
        "volatility": [0.6, 0.5, 0.5, 0.75, 0.7, 0.7, 0.9, 0.85, 0.85],
    }
)
config = CausalPanelConfig(
    unit_col="unit",
    time_col="time",
    outcome_col="outcome",
    treatment_col="treatment",
    post_treatment_col="post",
    covariate_cols=("volatility",),
    instrument_cols=("instrument",),
)
panel = assemble_causal_panel(entropy, market, config)
propensity = build_propensity_features(panel, config, window=2)
did = difference_in_differences(panel, config)
iv = two_stage_least_squares(panel, config)
sc = synthetic_control(panel, config, treated_unit="treated", donor_units=["control_a", "control_b"])
summary = build_summary_table(did, iv_result=iv, sc_result=sc)
scenario = PolicyScenario(
    name="Liquidity Injection",
    description="Add liquidity during crisis weeks",
    target_group="primary dealers",
)
print(summary)
print(format_policy_narrative(scenario, did))
